In [ ]:

import pandas as pd
import numpy as np
import requests

Langkah 3 & 4: Memuat Dataset dan Eksplorasi Awal

# File 'housing_dirty.csv' di upload lewat google drive

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data/housing_dirty.csv')

print("=== INFO DATASET ===")
df.info()

print("\n=== DESKRIPSI STATISTIK ===")
print(df.describe())

print("\n=== JUMLAH MISSING VALUES AWAL ===")
print(df.isnull().sum())

print('\nShape awal dataset:', df.shape)

=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            130 non-null    int64
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64
 6   kondisi       130 non-null    object
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

=== DESKRIPSI STATISTIK ===
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.250000
50%     65.500000   193.800000  6.550000e+02    4.000000   2002.000000
75%     97.750000   280.675000  9.550000e+02    5.000000   2011.750000
max    130.000000  9500.000000  1.000000e+08    6.000000   9999.000000

=== JUMLAH MISSING VALUES AWAL ===
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

Shape awal dataset: (130, 7)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Langkah 5: Menghapus Baris Duplikat

df.drop_duplicates(inplace=True)
print('Shape setelah hapus duplikat:', df.shape)

Shape setelah hapus duplikat: (130, 7)
Langkah 6: Normalisasi String (Kolom Kota & Kondisi)

df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print("Contoh data setelah normalisasi string:")
print(df[['kota', 'kondisi']].head())

Contoh data setelah normalisasi string:
    kota kondisi
0  Jogja    baik
1  Medan   bagus
2  Depok    baik
3    Ygy    baik
4  Medan  sedang
Langkah 7: Imputasi Missing Values

# Median untuk numerik (luas_m2 dan harga_juta)
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Modus untuk kategorik (kamar)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print("Jumlah missing values setelah imputasi:")
print(df.isnull().sum())

Jumlah missing values setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64
Langkah 8: Penanganan Outlier Menggunakan IQR Fence

# Berdasarkan instruksi langkah 8, penanganan pada kolom harga_juta dan luas_m2
for col in ['harga_juta', 'luas_m2']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # Menentukan batas bawah dan batas atas
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    # Melakukan clipping (pembatasan nilai ekstrem)
    df[col] = df[col].clip(batas_bawah, batas_atas)

print("Penanganan outlier selesai.")

Penanganan outlier selesai.
Langkah 9 & 10: Validasi Akhir dan Ekspor Dataset Bersih

# Validasi akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing values!'
assert df.duplicated().sum() == 0, 'Masih ada data duplikat!'

print('Shape akhir setelah dibersihkan:', df.shape)

# Ekspor ke file CSV baru
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih berhasil diekspor dengan nama "housing_clean.csv"!')

Shape akhir setelah dibersihkan: (130, 7)
Dataset bersih berhasil diekspor dengan nama "housing_clean.csv"!
Langkah 11: Akses API JSONPlaceholder dan Simpan Sebagai DataFrame

url_api = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url_api)

if response.status_code == 200:
    data_json = response.json()
    df_api = pd.DataFrame(data_json)
    print("Berhasil mengambil data dari API JSONPlaceholder!")
    print("\n5 Baris Pertama Data API:")
    print(df_api.head())
else:
    print(f"Gagal mengakses API. Status Code: {response.status_code}")

Berhasil mengambil data dari API JSONPlaceholder!

5 Baris Pertama Data API:
   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...
1       1   2                                       qui est esse
2       1   3  ea molestias quasi exercitationem repellat qui...
3       1   4                               eum et est occaecati
4       1   5                                 nesciunt quas odio

                                                body
0  quia et suscipit\nsuscipit recusandae consequu...
1  est rerum tempore vitae\nsequi sint nihil repr...
2  et iusto sed quo iure\nvoluptatem occaecati om...
3  ullam et saepe reiciendis voluptatem adipisci\...
4  repudiandae veniam quaerat sunt sed\nalias aut...